# Sensitivity Router Middleware Demo

This notebook demonstrates `SensitivityRouterMiddleware` integrated into a LangChain ReAct agent.

**What it shows:**

| Scenario | Expected behavior |
|----------|------------------|
| 1. Clean query | Default LLM is used |
| 2. Sensitive query (contains PII / credentials) | Routed to safe LLM |
| 3. RAG tool returns documents from a sensitive source path | All subsequent calls routed to safe LLM |

**Key classes used:**
- `SensitivityRouterMiddleware` — the middleware
- `SensitivityRouterConfig` — configuration (safe LLM, source patterns)
- `DefaultSensitivityScorer` — built-in hybrid scorer (regex + keywords + Presidio + heuristics)
- `DefaultScorerConfig` — scorer configuration (threshold, entity weights, …)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)
sys.path.append(".")

## 1. Build the sensitivity scorer

In [ ]:
from genai_tk.agents.langchain.middleware.presidio_detector import PresidioDetectorConfig
from genai_tk.agents.langchain.middleware.sensitivity_scorer import (
    DefaultScorerConfig,
    DefaultSensitivityScorer,
)

scorer_config = DefaultScorerConfig(
    sensitivity_threshold=0.30,  # moderate threshold
    detector=PresidioDetectorConfig(
        enable_spacy=True,
        spacy_model="en_core_web_sm",
    ),
)
scorer = DefaultSensitivityScorer(scorer_config)

# Quick manual tests
samples = [
    "What is the capital of France?",
    "My email is alice@example.com",
    "The root password is topsecret123",
    "Here is my IBAN: FR76 3000 6000 0112 3456 7890 189",
]

from rich.table import Table

table = Table(title="Scorer Demo")
table.add_column("Input", style="white", max_width=60)
table.add_column("Score", justify="right")
table.add_column("Level")
table.add_column("Sensitive?")

for text in samples:
    r = scorer.assess(text)
    color = {"low": "green", "medium": "yellow", "high": "orange3", "critical": "red"}.get(r.level, "white")
    table.add_row(
        text[:60],
        f"{r.score:.3f}",
        f"[{color}]{r.level}[/{color}]",
        "✓" if r.is_sensitive else "✗",
    )

print(table)

## 2. Build the router middleware

Configure it with:
- A **safe LLM** — a local/private model that handles sensitive data
- **Sensitive source patterns** — RAG document paths that trigger safe-LLM routing

In [ ]:
from genai_tk.agents.langchain.middleware.sensitivity_router_middleware import (
    SensitivityRouterConfig,
    SensitivityRouterMiddleware,
)
from genai_tk.core.llm_factory import get_llm

# Configure the safe LLM (use your local/private model tag here)
# Example: "ollama_local", "safe_model", or any model tag defined in baseline.yaml
SAFE_LLM_TAG = "default"  # change to a local model for real privacy

router_config = SensitivityRouterConfig(
    safe_llm=SAFE_LLM_TAG,
    sensitive_source_patterns=[
        "**/hr/**",
        "**/confidential/**",
        "**/personnel/**",
    ],
)

router_middleware = SensitivityRouterMiddleware(config=router_config)
# Override the internal scorer with our configured scorer
router_middleware._scorer = scorer

print("Router middleware created")
print(f"  Safe LLM: [green]{SAFE_LLM_TAG}[/green]")
print(f"  Sensitive source patterns: {router_config.sensitive_source_patterns}")

## 3. Scenario 1 — Clean query → default LLM

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"The weather in {city} is sunny, 22°C."


llm = get_llm()

agent = create_agent(
    model=llm,
    tools=[get_weather],
    middleware=[router_middleware],
    system_prompt="You are a helpful assistant.",
)

print("[bold cyan]Scenario 1: Clean query[/bold cyan]")
print("User: What is the weather in Berlin?")
print()

result = agent.invoke(
    {"messages": [HumanMessage(content="What is the weather in Berlin?")]},
    config={"configurable": {"thread_id": "scenario-1"}},
)

from genai_tk.agents.langchain.langchain_agent import _extract_content

print("[bold green]Response:[/bold green]", _extract_content(result))

## 4. Scenario 2 — Sensitive query → routed to safe LLM

In [ ]:
print("[bold yellow]Scenario 2: Sensitive query with credentials[/bold yellow]")
sensitive_query = "My server root password is Tr0ub4dor&3. Can you help me update it?"
print(f"User: {sensitive_query}")
print()

# Assess manually to show score
assessment = scorer.assess(sensitive_query)
print(
    f"[dim]Scorer result: score={assessment.score:.3f}, level={assessment.level}, sensitive={assessment.is_sensitive}[/dim]"
)
print()

result = agent.invoke(
    {"messages": [HumanMessage(content=sensitive_query)]},
    config={"configurable": {"thread_id": "scenario-2"}},
)

print("[bold green]Response (routed to safe LLM):[/bold green]", _extract_content(result))

## 5. Scenario 3 — RAG tool returns sensitive documents → sticky safe-LLM routing

In [ ]:
from langchain_core.documents import Document

# Create a fresh router for this scenario (to reset state)
router_3 = SensitivityRouterMiddleware(
    SensitivityRouterConfig(
        safe_llm=SAFE_LLM_TAG,
        sensitive_source_patterns=["**/hr/**"],
    )
)
router_3._scorer = scorer


@tool(response_format="content_and_artifact")
def search_hr_documents(query: str) -> tuple[str, list[Document]]:
    """Search HR documents for information."""
    docs = [
        Document(
            page_content=f"HR policy: {query} — see section 4.2 of the Employee Handbook.",
            metadata={"source": "/data/hr/employee_handbook.pdf", "page": 42},
        )
    ]
    return f"Found {len(docs)} document(s) about '{query}'", docs


agent_3 = create_agent(
    model=llm,
    tools=[search_hr_documents],
    middleware=[router_3],
    system_prompt="You are an HR assistant. Use search_hr_documents to answer HR questions.",
)

THREAD_3 = "scenario-3"

print("[bold yellow]Scenario 3: RAG query hitting sensitive HR documents[/bold yellow]")
print("User: What is the vacation policy?")
print()

result = agent_3.invoke(
    {"messages": [HumanMessage(content="What is the vacation policy?")]},
    config={"configurable": {"thread_id": THREAD_3}},
)

print("[bold green]Response:[/bold green]", _extract_content(result))
print()

# Check if sensitive source was recorded
sources = router_3._sensitive_sources.get(THREAD_3, set())
print(f"[dim]Sensitive sources recorded for thread '{THREAD_3}':[/dim]", sources)

In [ ]:
# Follow-up question — should also route to safe LLM (sticky)
print("[bold yellow]Follow-up question (should still route to safe LLM — sticky):[/bold yellow]")
print("User: What about parental leave?")

result_2 = agent_3.invoke(
    {"messages": [HumanMessage(content="What about parental leave?")]},
    config={"configurable": {"thread_id": THREAD_3}},
)

print("[bold green]Response:[/bold green]", _extract_content(result_2))

## 6. YAML configuration equivalent

```yaml
langchain_agents:
  profiles:
    - name: SecureAgent
      type: react
      llm: default
      middlewares:
        - class: genai_tk.agents.langchain.middleware.sensitivity_router_middleware:SensitivityRouterMiddleware
          safe_llm: ollama_local        # your private/local model tag
          sensitive_source_patterns:
            - "**/hr/**"
            - "**/confidential/**"
          # scorer_class: genai_tk.agents.langchain.middleware.sensitivity_scorer:DefaultSensitivityScorer
          # scorer_kwargs: {}  # uses defaults
```

Then run with:
```bash
uv run cli agents langchain --profile SecureAgent --chat
```

### Custom scorer

To use a custom scorer, implement the `SensitivityScorer` protocol:

```python
from genai_tk.agents.langchain.middleware.sensitivity_scorer import SensitivityScorer, SensitivityAssessment

class MyCustomScorer(SensitivityScorer):
    def assess(self, text: str) -> SensitivityAssessment:
        is_sensitive = "secret" in text.lower()
        score = 0.9 if is_sensitive else 0.1
        return SensitivityAssessment(
            is_sensitive=is_sensitive, score=score,
            level="high" if is_sensitive else "low",
            summary="Custom scorer result"
        )
```

Then reference it in YAML:
```yaml
scorer_class: myapp.scorers:MyCustomScorer
scorer_kwargs: {}
```